# Notebook 2 — Final Slow ConvLSTM Implementation
Based on improved experimental settings and encoder–forecaster ConvLSTM structure.
Includes stabilized training, kernel=5, better visuals, normalized losses.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import random

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)


In [ ]:
# ===== Final Slow Config =====

IMG_SIZE=32

IN_LEN=10
OUT_LEN=10

KERNEL_SIZE=5

HIDDEN_DIMS=[16,32]

TF_START=1.0
TF_END=0.4

EPOCHS=40

N_TRAIN=500
N_VAL=100

BATCH_SIZE=8


In [ ]:
# ===== Dataset =====

class MovingBlobDataset(Dataset):

    def __init__(self,n):

        self.data=[]

        for _ in range(n):

            seq=np.zeros(
                (IN_LEN+OUT_LEN,1,IMG_SIZE,IMG_SIZE)
            )

            x=np.random.randint(5,IMG_SIZE-5)
            y=np.random.randint(5,IMG_SIZE-5)

            dx=np.random.choice([-1,1])
            dy=np.random.choice([-1,1])

            for t in range(IN_LEN+OUT_LEN):

                seq[t,0,y-2:y+2,x-2:x+2]=1

                x+=dx
                y+=dy

                if x<5 or x>IMG_SIZE-5:
                    dx*=-1

                if y<5 or y>IMG_SIZE-5:
                    dy*=-1

            self.data.append(seq)

        self.data=np.array(self.data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):

        seq=torch.tensor(
            self.data[idx],
            dtype=torch.float32
        )

        return seq[:IN_LEN],seq[IN_LEN:]


In [ ]:
# ===== ConvLSTM Cell =====

class ConvLSTMCell(nn.Module):

    def __init__(self,input_dim,hidden_dim):

        super().__init__()

        self.hidden_dim=hidden_dim

        self.conv=nn.Conv2d(
            input_dim+hidden_dim,
            4*hidden_dim,
            kernel_size=KERNEL_SIZE,
            padding=KERNEL_SIZE//2
        )

    def forward(self,x,h,c):

        combined=torch.cat([x,h],dim=1)

        conv_out=self.conv(combined)

        i,f,o,g=torch.split(
            conv_out,
            self.hidden_dim,
            dim=1
        )

        i=torch.sigmoid(i)
        f=torch.sigmoid(f)
        o=torch.sigmoid(o)
        g=torch.tanh(g)

        c=f*c+i*g
        h=o*torch.tanh(c)

        return h,c


In [ ]:
# ===== Encoder–Forecaster =====

class EncoderForecaster(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder_cells=nn.ModuleList()
        self.forecaster_cells=nn.ModuleList()

        input_dim=1

        for hidden in HIDDEN_DIMS:

            self.encoder_cells.append(
                ConvLSTMCell(input_dim,hidden)
            )

            self.forecaster_cells.append(
                ConvLSTMCell(input_dim,hidden)
            )

            input_dim=hidden

        self.output_conv=nn.Conv2d(
            HIDDEN_DIMS[-1],
            1,
            kernel_size=1
        )

    def forward(self,x,y=None,tf_ratio=0):

        B,T,C,H,W=x.shape

        h=[]
        c=[]

        for hidden in HIDDEN_DIMS:

            h.append(torch.zeros(B,hidden,H,W,device=x.device))
            c.append(torch.zeros_like(h[-1]))

        # Encoder

        for t in range(T):

            inp=x[:,t]

            for l,cell in enumerate(self.encoder_cells):

                h[l],c[l]=cell(inp,h[l],c[l])
                inp=h[l]

        outputs=[]

        cur_input=x[:,-1]

        # Forecaster

        for t in range(OUT_LEN):

            inp=cur_input

            for l,cell in enumerate(self.forecaster_cells):

                h[l],c[l]=cell(inp,h[l],c[l])
                inp=h[l]

            out=self.output_conv(h[-1])

            outputs.append(out)

            if y is not None:

                if random.random()<tf_ratio:
                    cur_input=y[:,t]
                else:
                    cur_input=out

            else:

                cur_input=out

        outputs=torch.stack(outputs,dim=1)

        return outputs


In [ ]:
# ===== Data Loaders =====

train_ds=MovingBlobDataset(N_TRAIN)
val_ds=MovingBlobDataset(N_VAL)

train_loader=DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader=DataLoader(
    val_ds,
    batch_size=BATCH_SIZE
)


In [ ]:
# ===== Training =====

model=EncoderForecaster().to(device)

optimizer=optim.Adam(
    model.parameters(),
    lr=1e-3
)

criterion=nn.MSELoss()

train_losses=[]
val_losses=[]

best_val=float("inf")

for epoch in range(EPOCHS):

    tf_ratio=max(
        TF_END,
        TF_START - epoch*(TF_START-TF_END)/EPOCHS
    )

    model.train()

    train_loss=0

    for x,y in train_loader:

        x=x.to(device)
        y=y.to(device)

        optimizer.zero_grad()

        pred=model(x,y,tf_ratio)

        loss=criterion(pred,y)

        loss.backward()

        optimizer.step()

        train_loss+=loss.item()

    train_loss/=len(train_loader)

    model.eval()

    val_loss=0

    with torch.no_grad():

        for x,y in val_loader:

            x=x.to(device)
            y=y.to(device)

            pred=model(x)

            loss=criterion(pred,y)

            val_loss+=loss.item()

    val_loss/=len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss<best_val:

        best_val=val_loss

        torch.save(
            model.state_dict(),
            "best_slow_model.pt"
        )

    print(
        f"Epoch {epoch+1}/{EPOCHS} "
        f"Train {train_loss:.4f} "
        f"Val {val_loss:.4f} "
        f"TF {tf_ratio:.2f}"
    )


In [ ]:
# ===== Load Best =====

model.load_state_dict(
    torch.load("best_slow_model.pt")
)

model.eval()


In [ ]:
# ===== Training Curve =====

plt.figure(figsize=(6,4))

plt.plot(train_losses,label="Train")
plt.plot(val_losses,label="Val")

plt.title("Training Convergence")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()

plt.show()


In [ ]:
# ===== Prediction Visualisation =====

x,y=val_ds[0]

with torch.no_grad():

    pred=model(
        x.unsqueeze(0).to(device)
    ).cpu().squeeze(0)

plt.figure(figsize=(12,4))

for i in range(OUT_LEN):

    plt.subplot(2,OUT_LEN,i+1)

    plt.imshow(
        y[i,0],
        cmap="inferno",
        vmin=0,
        vmax=1
    )

    plt.axis("off")

    plt.subplot(2,OUT_LEN,i+1+OUT_LEN)

    plt.imshow(
        pred[i,0],
        cmap="inferno",
        vmin=0,
        vmax=1
    )

    plt.axis("off")

plt.suptitle("Ground Truth vs Prediction")

plt.show()


In [ ]:
# ===== Error Heatmap =====

error=torch.abs(pred-y)

plt.figure(figsize=(12,2))

for i in range(OUT_LEN):

    plt.subplot(1,OUT_LEN,i+1)

    plt.imshow(
        error[i,0],
        cmap="hot",
        vmin=0,
        vmax=1
    )

    plt.axis("off")

plt.suptitle("Prediction Error Growth")

plt.show()
